## Configuração do ambiente (local)

**Objetivo:** Explicabilidade (SHAP): XGBoost (TreeExplainer) e CNN (GradientExplainer).

**Entradas:** `Data/processed/dataset_abide_processado.pkl` (07), `Models/08_modelos_baselines.joblib` (08), `Models/09_cnn_customizada_fold0.keras` (09), atlas CC200 (baixado só se faltar).

**Processamento:** SHAP summary (top 20), importância por ROI, grafo do top 2% de conexões, mapa SHAP da CNN.

**Saídas:** `Figures/10_*.png`.

In [ ]:
# ------------------------------------------------------------
# Configuração do projeto (execução local — VS Code)
# Garante que a raiz do projeto esteja no sys.path, funcionando tanto se o
# diretório de trabalho do kernel for notebooks/ quanto a raiz do projeto.
# ------------------------------------------------------------
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

In [ ]:
import shap
print('SHAP versão:', shap.__version__)

## Parte 1 — SHAP sobre o XGBoost (TreeExplainer)

In [ ]:
exigir_arquivo(PATHS['processed'] / 'dataset_abide_processado.pkl', '07_Preparacao_Dados_ABIDE.ipynb')
exigir_arquivo(PATHS['models'] / '08_modelos_baselines.joblib', '08_Baselines_ABIDE.ipynb')

import pickle
import joblib
import numpy as np
import pandas as pd

with open(os.path.join(PATHS['processed'], 'dataset_abide_processado.pkl'), 'rb') as f:
    dataset = pickle.load(f)
X, y, groups, folds = dataset['X'], dataset['y'], dataset['groups'], dataset['folds']

modelos_baseline = joblib.load(os.path.join(PATHS['models'], '08_modelos_baselines.joblib'))

# Usa o fold 0 do XGBoost como referência para a explicação
chave_xgb = 'XGBoost_fold0'
if chave_xgb not in modelos_baseline:
    chave_xgb = [k for k in modelos_baseline if k.startswith('XGBoost')][0]
    print(f'Aviso: "XGBoost_fold0" não encontrado, usando "{chave_xgb}" em seu lugar.')

modelo_xgb = modelos_baseline[chave_xgb]['modelo']
scaler_xgb = modelos_baseline[chave_xgb]['scaler']

fold_idx = int(chave_xgb.split('fold')[-1])
_, test_idx = folds[fold_idx]
X_test_scaled = scaler_xgb.transform(X[test_idx])

print('Explicando', X_test_scaled.shape[0], 'sujeitos de teste (fold', fold_idx, ') com o XGBoost.')


In [ ]:
explainer_xgb = shap.TreeExplainer(modelo_xgb)
shap_values_xgb = explainer_xgb.shap_values(X_test_scaled)

print('Shape dos valores SHAP:', np.array(shap_values_xgb).shape)


In [ ]:
import matplotlib.pyplot as plt

# Nomes das features: cada uma é um par de ROIs (i, j) do triângulo superior
n_rois = 200
triu_idx = np.triu_indices(n_rois, k=1)
nomes_features = [f'ROI{i}-ROI{j}' for i, j in zip(*triu_idx)]

fig = plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values_xgb, X_test_scaled, feature_names=nomes_features,
                   max_display=20, show=False)
plt.title('SHAP — Conexões mais importantes para o XGBoost (top 20)')
plt.tight_layout()
plt.savefig(os.path.join(PATHS['figures'], '10_shap_summary_xgboost.png'), dpi=300, bbox_inches='tight')
plt.show()


## Parte 2 — Mapa de importância por ROI sobre o cérebro


In [ ]:
import urllib.request
import nibabel as nib
from nilearn import plotting
from nilearn.plotting import find_parcellation_cut_coords

# Reaproveita o atlas já baixado nos notebooks 03/04 (baixa de novo só se faltar)
atlas_path = os.path.join(PATHS['raw'], 'cc200_roi_atlas.nii.gz')
if not os.path.exists(atlas_path):
    atlas_url = 'https://fcp-indi.s3.amazonaws.com/data/Projects/ABIDE_Initiative/Resources/cc200_roi_atlas.nii.gz'
    urllib.request.urlretrieve(atlas_url, atlas_path)

coords, _ = find_parcellation_cut_coords(atlas_path, return_label_names=True)
print('Coordenadas de', coords.shape[0], 'ROIs.')

# Importância média |SHAP| por ROI, agregando todas as conexões que a envolvem
shap_medio = np.abs(shap_values_xgb).mean(axis=0)   # (n_features,) -> uma por conexão

importancia_roi = np.zeros(n_rois)
contagem_roi = np.zeros(n_rois)
for idx, (i, j) in enumerate(zip(*triu_idx)):
    importancia_roi[i] += shap_medio[idx]
    importancia_roi[j] += shap_medio[idx]
    contagem_roi[i] += 1
    contagem_roi[j] += 1
importancia_roi = importancia_roi / np.maximum(contagem_roi, 1)  # média, evita viés por grau

n_min = min(len(coords), n_rois)

fig = plt.figure(figsize=(10, 5))
plotting.plot_markers(
    importancia_roi[:n_min], coords[:n_min],
    node_size=importancia_roi[:n_min] / importancia_roi[:n_min].max() * 200,
    title='Importância média |SHAP| por ROI (XGBoost)',
    display_mode='ortho',
    figure=fig,
)
plt.savefig(os.path.join(PATHS['figures'], '10_shap_importancia_roi_cerebro.png'), dpi=300, bbox_inches='tight')
plt.show()


## Parte 3 — Grafo das conexões mais discriminativas


In [ ]:
# Migração: edge_threshold agora é float (o original passava string numérica sem '%',
# que o nilearn rejeita com ValueError). Limiar = percentil 98 de |SHAP| (top 2%).
shap_matriz = np.zeros((n_rois, n_rois))
for idx, (i, j) in enumerate(zip(*triu_idx)):
    shap_matriz[i, j] = shap_medio[idx]
    shap_matriz[j, i] = shap_medio[idx]

limiar_shap = np.percentile(shap_medio, 98)  # top 2% conexões mais discriminativas

fig = plt.figure(figsize=(10, 5))
plotting.plot_connectome(
    shap_matriz[:n_min, :n_min], coords[:n_min],
    edge_threshold=float(limiar_shap),
    node_size=15,
    title='Conexões mais discriminativas (top 2% por |SHAP|) — XGBoost',
    display_mode='ortho',
    colorbar=True,
    figure=fig,
)
plt.savefig(os.path.join(PATHS['figures'], '10_shap_grafo_conexoes_discriminativas.png'), dpi=300, bbox_inches='tight')
plt.show()


## Parte 4 — SHAP sobre a CNN (GradientExplainer)


In [ ]:
import tensorflow as tf

caminho_cnn_fold0 = os.path.join(PATHS['models'], '09_cnn_customizada_fold0.keras')
if os.path.exists(caminho_cnn_fold0):
    modelo_cnn = tf.keras.models.load_model(caminho_cnn_fold0)

    with open(os.path.join(PATHS['conectivity'], 'connectivity_matrices_abide_validado.pkl'), 'rb') as f:
        conn_dict = pickle.load(f)
    sub_ids_img = dataset['sub_ids']
    matrices_01 = (np.array([conn_dict[s] for s in sub_ids_img]).astype('float32') + 1.0) / 2.0
    X_img_1ch = matrices_01[..., np.newaxis]

    _, test_idx_0 = folds[0]
    X_background = X_img_1ch[folds[0][0]][:50]   # amostra do treino do fold 0, como referência (background)
    X_explicar = X_img_1ch[test_idx_0][:10]      # explica os 10 primeiros sujeitos de teste do fold 0

    explainer_cnn = shap.GradientExplainer(modelo_cnn, X_background)
    shap_values_cnn = explainer_cnn.shap_values(X_explicar)

    mapa_medio = np.abs(np.array(shap_values_cnn)).mean(axis=(0, -1)).squeeze()
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(mapa_medio, cmap='hot')
    ax.set_title('Importância |SHAP| média por conexão — CNN (fold 0, 10 sujeitos)')
    plt.colorbar(im, ax=ax, label='|SHAP| médio')
    plt.tight_layout()
    plt.savefig(os.path.join(PATHS['figures'], '10_shap_cnn_mapa_conexoes.png'), dpi=300, bbox_inches='tight')
    plt.show()
else:
    print(f'Modelo não encontrado em {caminho_cnn_fold0} — rode o notebook 09 antes desta célula.')
    print('Pulando a explicabilidade da CNN.')
